In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
SINKRONISASI KATALOG INDONESIA DENGAN JSON WAVEFORM (REVISI)
- Menambahkan origin_time, latitude, longitude, magnitude ke metadata
- Menggunakan timestamp dari nama file
- Menangani format datetime ISO8601 dengan milidetik dan timezone
"""

import json
import pandas as pd
import re
import os
from datetime import datetime

# =============================================
# 1. KONFIGURASI
# =============================================
JSON_INDONESIA_PATH = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_3c_4_SYNCED.json'
CATALOG_INDONESIA_PATH = "/Volumes/Extreme SSD/katalog/hybrid_catalog_filtered.csv"
JSON_OUTPUT_PATH = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_3c_4_SYNCED.json'

# =============================================
# 2. FUNGSI BANTUAN
# =============================================

def extract_timestamp_from_key(key):
    """
    Ekstrak timestamp YYYYMMDD_HHMMSS dari key JSON.
    Format: NET_STA_YYYYMMDD_HHMMSS
    """
    match = re.search(r'(\d{8})_(\d{6})', key)
    if match:
        return f"{match.group(1)}_{match.group(2)}"
    match = re.search(r'(\d{14})', key)
    if match:
        ts = match.group(1)
        return f"{ts[:8]}_{ts[8:]}"
    return None

def read_catalog(csv_path):
    """Baca katalog dengan parsing datetime robust."""
    df = pd.read_csv(csv_path)
    print(f"📂 Katalog dimuat: {len(df)} baris")
    print(f"📋 Kolom: {df.columns.tolist()}")
    
    # Deteksi kolom
    time_col = None
    for col in df.columns:
        if 'time' in col.lower() or 'datetime' in col.lower() or 'origin' in col.lower():
            time_col = col
            break
    if time_col is None:
        raise ValueError("❌ Kolom waktu tidak ditemukan.")
    print(f"🕒 Kolom waktu: '{time_col}'")
    
    lat_col = next((col for col in df.columns if 'lat' in col.lower()), None)
    lon_col = next((col for col in df.columns if 'lon' in col.lower()), None)
    mag_col = next((col for col in df.columns if 'mag' in col.lower()), None)
    
    print(f"🌐 Kolom lat: '{lat_col}', lon: '{lon_col}'")
    if mag_col:
        print(f"📏 Kolom magnitude: '{mag_col}'")
    
    # ===== PERBAIKAN: Parsing datetime =====
    # Coba dengan format ISO8601 (mengandung milidetik dan timezone)
    try:
        df['datetime'] = pd.to_datetime(df[time_col], utc=True, format='ISO8601')
        print("✅ Parsing datetime dengan format ISO8601 berhasil.")
    except Exception as e:
        print(f"⚠️ Format ISO8601 gagal: {e}")
        print("   Mencoba dengan infer_datetime_format=True...")
        try:
            df['datetime'] = pd.to_datetime(df[time_col], utc=True, infer_datetime_format=True)
            print("✅ Parsing datetime dengan infer berhasil.")
        except Exception as e2:
            print(f"❌ Gagal memparse datetime: {e2}")
            raise
    
    # Buang baris yang tidak terkonversi
    initial_len = len(df)
    df = df.dropna(subset=['datetime'])
    if len(df) < initial_len:
        print(f"⚠️ {initial_len - len(df)} baris gagal diparse dan di-drop.")
    
    # Buat timestamp key
    df['timestamp_key'] = df['datetime'].dt.strftime("%Y%m%d_%H%M%S")
    
    # Hapus duplikat timestamp (pertahankan yang pertama)
    dup_count = df.duplicated(subset=['timestamp_key']).sum()
    if dup_count > 0:
        print(f"⚠️ Ditemukan {dup_count} duplikat timestamp. Hanya yang pertama dipertahankan.")
        df = df.drop_duplicates(subset=['timestamp_key'], keep='first')
    
    print(f"✅ Total event setelah parsing dan deduplikasi: {len(df)}")
    
    return df, lat_col, lon_col, mag_col

def sync_catalog(json_path, df_catalog, lat_col, lon_col, mag_col, output_path):
    """Sinkronkan JSON dengan katalog."""
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    print(f"📂 Total entri di JSON: {len(data)}")
    
    # Buat dictionary katalog
    catalog_dict = {}
    for _, row in df_catalog.iterrows():
        key = row['timestamp_key']
        catalog_dict[key] = {
            'origin_time': row['datetime'].isoformat(),
            'latitude': row[lat_col],
            'longitude': row[lon_col],
            'magnitude': row[mag_col] if mag_col else None
        }
    
    print(f"📂 Total timestamp unik di katalog: {len(catalog_dict)}")
    
    # Sinkronisasi
    matched = 0
    unmatched = 0
    updated_json = {}
    
    for key, record in data.items():
        ts = extract_timestamp_from_key(key)
        if ts is None:
            unmatched += 1
            updated_json[key] = record
            continue
        
        if ts in catalog_dict:
            matched += 1
            cat_info = catalog_dict[ts]
            # Tambahkan metadata
            if 'metadata' not in record:
                record['metadata'] = {}
            record['metadata']['origin_time'] = cat_info['origin_time']
            record['metadata']['latitude'] = cat_info['latitude']
            record['metadata']['longitude'] = cat_info['longitude']
            if cat_info['magnitude'] is not None:
                record['metadata']['magnitude'] = float(cat_info['magnitude'])
            updated_json[key] = record
        else:
            unmatched += 1
            updated_json[key] = record
    
    print(f"✅ Match: {matched}")
    print(f"❌ Unmatch: {unmatched}")
    
    # Simpan
    with open(output_path, 'w') as f:
        json.dump(updated_json, f, indent=2)
    
    return matched, unmatched

# =============================================
# 3. MAIN
# =============================================

if __name__ == "__main__":
    print("="*70)
    print("🔗 SINKRONISASI KATALOG INDONESIA (REVISI)")
    print("="*70)
    
    # Validasi path
    if not os.path.exists(JSON_INDONESIA_PATH):
        print(f"❌ JSON tidak ditemukan: {JSON_INDONESIA_PATH}")
        exit(1)
    
    if not os.path.exists(CATALOG_INDONESIA_PATH):
        print(f"❌ Katalog tidak ditemukan: {CATALOG_INDONESIA_PATH}")
        exit(1)
    
    # Baca katalog
    try:
        df, lat_col, lon_col, mag_col = read_catalog(CATALOG_INDONESIA_PATH)
    except Exception as e:
        print(f"❌ Gagal membaca katalog: {e}")
        exit(1)
    
    # Sinkronisasi
    matched, unmatched = sync_catalog(
        JSON_INDONESIA_PATH,
        df,
        lat_col,
        lon_col,
        mag_col,
        JSON_OUTPUT_PATH
    )
    
    print("\n" + "="*70)
    print("✅ SINKRONISASI SELESAI!")
    print(f"📂 Output: {JSON_OUTPUT_PATH}")
    print(f"   Match: {matched}")
    print(f"   Unmatch: {unmatched}")
    print("="*70)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
SINKRONISASI KATALOG INDONESIA DENGAN JSON WAVEFORM 1C
- Menambahkan origin_time, latitude, longitude, magnitude ke metadata
- Menggunakan timestamp dari key JSON (NET_STA_YYYYMMDD_HHMMSS)
- Khusus untuk file JSON 1C (hanya komponen Z)
"""

import json
import pandas as pd
import re
import os
from datetime import datetime

# =============================================
# 1. KONFIGURASI (UBAH SESUAI ANDA)
# =============================================
JSON_1C_INPUT = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_1c_4.json'
CATALOG_CSV = "/Volumes/Extreme SSD/katalog/hybrid_catalog_filtered.csv"
JSON_1C_OUTPUT = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_1c_4_SYNCED.json'

# =============================================
# 2. FUNGSI BANTUAN
# =============================================

def extract_timestamp_from_key(key):
    """
    Ekstrak timestamp YYYYMMDD_HHMMSS dari key JSON.
    Format: NET_STA_YYYYMMDD_HHMMSS atau YYYYMMDD_HHMMSS
    """
    # Pola 1: 8 digit underscore 6 digit
    match = re.search(r'(\d{8})_(\d{6})', key)
    if match:
        return f"{match.group(1)}_{match.group(2)}"
    # Pola 2: 14 digit angka
    match = re.search(r'(\d{14})', key)
    if match:
        ts = match.group(1)
        return f"{ts[:8]}_{ts[8:]}"
    return None

def read_catalog(csv_path):
    """Baca katalog dengan parsing datetime robust."""
    df = pd.read_csv(csv_path)
    print(f"📂 Katalog dimuat: {len(df)} baris")
    print(f"📋 Kolom: {df.columns.tolist()}")
    
    # Deteksi kolom
    time_col = None
    for col in df.columns:
        if 'time' in col.lower() or 'datetime' in col.lower() or 'origin' in col.lower():
            time_col = col
            break
    if time_col is None:
        raise ValueError("❌ Kolom waktu tidak ditemukan.")
    print(f"🕒 Kolom waktu: '{time_col}'")
    
    lat_col = next((col for col in df.columns if 'lat' in col.lower()), None)
    lon_col = next((col for col in df.columns if 'lon' in col.lower()), None)
    mag_col = next((col for col in df.columns if 'mag' in col.lower()), None)
    
    print(f"🌐 Kolom lat: '{lat_col}', lon: '{lon_col}'")
    if mag_col:
        print(f"📏 Kolom magnitude: '{mag_col}'")
    
    # Parsing datetime
    try:
        df['datetime'] = pd.to_datetime(df[time_col], utc=True, format='ISO8601')
        print("✅ Parsing datetime dengan format ISO8601 berhasil.")
    except Exception as e:
        print(f"⚠️ Format ISO8601 gagal: {e}")
        print("   Mencoba dengan infer_datetime_format=True...")
        try:
            df['datetime'] = pd.to_datetime(df[time_col], utc=True, infer_datetime_format=True)
            print("✅ Parsing datetime dengan infer berhasil.")
        except Exception as e2:
            print(f"❌ Gagal memparse datetime: {e2}")
            raise
    
    # Buang baris yang tidak terkonversi
    initial_len = len(df)
    df = df.dropna(subset=['datetime'])
    if len(df) < initial_len:
        print(f"⚠️ {initial_len - len(df)} baris gagal diparse dan di-drop.")
    
    # Buat timestamp key
    df['timestamp_key'] = df['datetime'].dt.strftime("%Y%m%d_%H%M%S")
    
    # Hapus duplikat timestamp (pertahankan yang pertama)
    dup_count = df.duplicated(subset=['timestamp_key']).sum()
    if dup_count > 0:
        print(f"⚠️ Ditemukan {dup_count} duplikat timestamp. Hanya yang pertama dipertahankan.")
        df = df.drop_duplicates(subset=['timestamp_key'], keep='first')
    
    print(f"✅ Total event setelah parsing dan deduplikasi: {len(df)}")
    
    return df, lat_col, lon_col, mag_col

def sync_catalog_1c(json_path, df_catalog, lat_col, lon_col, mag_col, output_path):
    """Sinkronkan JSON 1C dengan katalog."""
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    print(f"📂 Total entri di JSON 1C: {len(data)}")
    
    # Buat dictionary katalog
    catalog_dict = {}
    for _, row in df_catalog.iterrows():
        key = row['timestamp_key']
        catalog_dict[key] = {
            'origin_time': row['datetime'].isoformat(),
            'latitude': row[lat_col],
            'longitude': row[lon_col],
            'magnitude': row[mag_col] if mag_col else None
        }
    
    print(f"📂 Total timestamp unik di katalog: {len(catalog_dict)}")
    
    # Sinkronisasi
    matched = 0
    unmatched = 0
    updated_json = {}
    
    for key, record in data.items():
        ts = extract_timestamp_from_key(key)
        if ts is None:
            unmatched += 1
            updated_json[key] = record
            continue
        
        if ts in catalog_dict:
            matched += 1
            cat_info = catalog_dict[ts]
            # Tambahkan metadata
            if 'metadata' not in record:
                record['metadata'] = {}
            record['metadata']['origin_time'] = cat_info['origin_time']
            record['metadata']['latitude'] = cat_info['latitude']
            record['metadata']['longitude'] = cat_info['longitude']
            if cat_info['magnitude'] is not None:
                record['metadata']['magnitude'] = float(cat_info['magnitude'])
            updated_json[key] = record
        else:
            unmatched += 1
            updated_json[key] = record
    
    print(f"✅ Match: {matched}")
    print(f"❌ Unmatch: {unmatched}")
    
    # Simpan
    with open(output_path, 'w') as f:
        json.dump(updated_json, f, indent=2)
    
    return matched, unmatched

# =============================================
# 3. MAIN
# =============================================

if __name__ == "__main__":
    print("="*70)
    print("🔗 SINKRONISASI KATALOG UNTUK JSON 1C")
    print("="*70)
    
    # Validasi path
    if not os.path.exists(JSON_1C_INPUT):
        print(f"❌ JSON 1C tidak ditemukan: {JSON_1C_INPUT}")
        exit(1)
    
    if not os.path.exists(CATALOG_CSV):
        print(f"❌ Katalog tidak ditemukan: {CATALOG_CSV}")
        exit(1)
    
    # Baca katalog
    try:
        df, lat_col, lon_col, mag_col = read_catalog(CATALOG_CSV)
    except Exception as e:
        print(f"❌ Gagal membaca katalog: {e}")
        exit(1)
    
    # Sinkronisasi
    matched, unmatched = sync_catalog_1c(
        JSON_1C_INPUT,
        df,
        lat_col,
        lon_col,
        mag_col,
        JSON_1C_OUTPUT
    )
    
    print("\n" + "="*70)
    print("✅ SINKRONISASI JSON 1C SELESAI!")
    print(f"📂 Output: {JSON_1C_OUTPUT}")
    print(f"   Match: {matched}")
    print(f"   Unmatch: {unmatched}")
    print("="*70)